# Installs and Imports

In [ ]:
%%capture
!git clone https://github.com/smartbugs/smartbugs-results.git
!git clone https://github.com/smartbugs/smartbugs-wild.git
!pip install datasets

Cloning into 'smartbugs-results'...
remote: Enumerating objects: 1109622, done.
remote: Total 1109622 (delta 0), reused 0 (delta 0), pack-reused 1109622 (from 1)
Receiving objects: 100% (1109622/1109622), 540.99 MiB | 9.95 MiB/s, done.
Resolving deltas: 100% (502462/502462), done.
Updating files: 100% (757583/757583), done.
Cloning into 'smartbugs-wild'...
remote: Enumerating objects: 47556, done.
remote: Counting objects: 100% (17866/17866), done.
remote: Compressing objects: 100% (7909/7909), done.
remote: Total 47556 (delta 14971), reused 9957 (delta 9957), pack-reused 29690 (from 1)
Receiving objects: 100% (47556/47556), 235.00 MiB | 16.43 MiB/s, done.
Resolving deltas: 100% (20505/20505), done.
Updating files: 100% (47407/47407), done.


In [ ]:
import json
from pprint import pprint
from collections import defaultdict
import pandas as pd
import os
from tqdm import tqdm
import re
from copy import deepcopy
from pathlib import Path
import glob
from multiprocessing import Pool as ThreadPool
from functools import partial
import os
import random
from google.colab import drive

# Tools

In [ ]:
tool_list = ['slither', 'mythril', 'smartcheck']
tool_sc = []
for tool in tool_list:
    tool_sc.append(os.listdir(f'/content/smartbugs-results/results/{tool}/icse20'))

In [ ]:
da_path = "/content/smartbugs-results/results/{}/icse20/{}/result.json"

with open(da_path.format('smartcheck', "0xb5fc7a96144dc3318cf8df6bebc5c5c8c563fb73")) as f:
    data = json.load(f)

pprint(data['analysis'])

[{'column': 25,
  'content': '0xAa1ae5e57dc05981D83eC7FcA0b3c7ee2565B7D6',
  'line': 79,
  'name': 'SOLIDITY_ADDRESS_HARDCODED',
  'patternId': 'adc165',
  'severity': 1},
 {'column': 18,
  'content': '0x22a740cC50365dBD6708B1B280829985a03b3C21',
  'line': 80,
  'name': 'SOLIDITY_ADDRESS_HARDCODED',
  'patternId': 'adc165',
  'severity': 1},
 {'column': 2,
  'content': 'functionbalanceOf(address_owner)publicconstantreturns(uint256balance);',
  'line': 32,
  'name': 'SOLIDITY_DEPRECATED_CONSTRUCTIONS',
  'patternId': '28fa69',
  'severity': 1},
 {'column': 2,
  'content': 'functiontokenBalance()constantpublicreturns(uint256){returntoken_reward.balanceOf(this);}',
  'line': 83,
  'name': 'SOLIDITY_DEPRECATED_CONSTRUCTIONS',
  'patternId': '28fa69',
  'severity': 1},
 {'column': 2,
  'content': 'functionlockOver()constantpublicreturns(bool){uint256current_time=now;returncurrent_time>end_time;}',
  'line': 95,
  'name': 'SOLIDITY_DEPRECATED_CONSTRUCTIONS',
  'patternId': '28fa69',
  'sever

In [ ]:
with open("/content/smartbugs-wild/contracts/0xb5fc7a96144dc3318cf8df6bebc5c5c8c563fb73.sol") as f:
    print(f.read())

pragma solidity ^0.4.18;

/**
* @title SafeMath
* @dev Math operations with safety checks that throw on error
*/
library SafeMath {
  function mul(uint256 a, uint256 b) internal pure returns (uint256) {
    uint256 c = a * b;
    assert(a == 0 || c / a == b);
    return c;
  }
  function div(uint256 a, uint256 b) internal pure returns (uint256) {
    // assert(b > 0); // Solidity automatically throws when dividing by 0
    uint256 c = a / b;
    // assert(a == b * c + a % b); // There is no case in which this doesn't hold
    return c;
  }
  function sub(uint256 a, uint256 b) internal pure returns (uint256) {
    assert(b <= a);
    return a - b;
  }
  function add(uint256 a, uint256 b) internal pure returns (uint256) {
    uint256 c = a + b;
    assert(c >= a);
    return c;
  }
}

contract token {

  function balanceOf(address _owner) public constant returns (uint256 balance);
  function transfer(address _to, uint256 _value) public returns (bool success);

}

contract Ownable {
  add

In [ ]:
def remove_comments(solidity_code):
    # Regex patterns to match single-line and multi-line comments
    single_line_comment_pattern = r"//.*?(?=\n|$)"
    multi_line_comment_pattern = r"/\*.*?\*/"

    # Remove single-line comments
    code_without_single_line_comments = re.sub(single_line_comment_pattern, '', solidity_code, flags=re.DOTALL)

    # Remove multi-line comments
    cleaned_code = re.sub(multi_line_comment_pattern, '', code_without_single_line_comments, flags=re.DOTALL)

    return cleaned_code.strip()

In [ ]:
# @title Get Sol lines and whole file
with open(f"/content/smartbugs-wild/contracts/{tool_sc[1][0]}.sol", "r", encoding='utf-8') as f:
    lines = f.readlines()
    sol_file = "".join(lines)


# print(lines)

sol_file = remove_comments(sol_file)
print(sol_file)
# print(sol_file.splitlines(keepends=True))
lines = sol_file.splitlines(keepends=True)
tmp = []
for line in lines:
  if line.strip() == '':
    continue
  tmp.append(line)
lines = tmp

pragma solidity 0.4.24;


contract Ownable {
  address public owner;


  event OwnershipRenounced(address indexed previousOwner);
  event OwnershipTransferred(
    address indexed previousOwner,
    address indexed newOwner
  );


  
  constructor() public {
    owner = msg.sender;
  }

  
  modifier onlyOwner() {
    require(msg.sender == owner);
    _;
  }

  
  function renounceOwnership() public onlyOwner {
    emit OwnershipRenounced(owner);
    owner = address(0);
  }

  
  function transferOwnership(address _newOwner) public onlyOwner {
    _transferOwnership(_newOwner);
  }

  
  function _transferOwnership(address _newOwner) internal {
    require(_newOwner != address(0));
    emit OwnershipTransferred(owner, _newOwner);
    owner = _newOwner;
  }
}



library ECRecovery {

  
  function recover(bytes32 hash, bytes sig)
    internal
    pure
    returns (address)
  {
    bytes32 r;
    bytes32 s;
    uint8 v;

    
    if (sig.length != 65) {
      return (address(0));
    }



In [ ]:
# @title Get source code functions
def read_solidity_file(filepath):
    with open(filepath, 'r', encoding='utf-8') as file:
        return file.readlines()

def find_function(lines, error_line):
    function_pattern = re.compile(r'\bfunction\b')
    modifier_pattern = re.compile(r'\bmodifier\b')
    constructor_pattern = re.compile(r'\bconstructor\b')
    start_line = None

    for i in range(error_line - 1, -1, -1):
        if function_pattern.search(lines[i]) or modifier_pattern.search(lines[i]) or constructor_pattern.search(lines[i]):
            start_line = i
            break

    if start_line is None:
        raise Exception("Function definition not found.")

    end_line = None
    brace_count = 0
    in_function = False

    for i in range(start_line, len(lines)):
        line = lines[i]
        brace_count += line.count('{')
        brace_count -= line.count('}')

        if brace_count > 0:
            in_function = True
        elif brace_count == 0 and in_function:
            end_line = i
            break


    if end_line is None:
        raise Exception("Function end not found.")

    function_code = "".join(lines[start_line:end_line + 1])
    return function_code

# # Example usage
# sol_file_path = f"/content/smartbugs-wild/contracts/{tool_sc[0][1]}.sol"
# src_lines = read_solidity_file(sol_file_path)

# function_codes = []

# for i in range(len(src_lines)):
#     if 'function' in src_lines[i]:
#         function = find_function(src_lines, i + 1)
#         if function.count('function') == 1 and function.count('Interface') == 0:
#             function_codes.append(function)

# for function_code in function_codes:
#     print(f"Function Code:\n{function_code.strip()}\n")
#     print("-" * 100)

In [ ]:
# @title Get Sol Functions
def read_solidity_file(filepath):
    with open(filepath, 'r', encoding='utf-8') as file:
        return file.readlines()

def find_function_smartcheck(lines, error_line):
    function_pattern = re.compile(r'\bfunction\b')
    modifier_pattern = re.compile(r'\bmodifier\b')
    constructor_pattern = re.compile(r'\bconstructor\b')
    start_line = None

    for i in range(error_line - 1, -1, -1):
        if function_pattern.search(lines[i]) or modifier_pattern.search(lines[i]) or constructor_pattern.search(lines[i]):
            start_line = i
            break

    if start_line is None:
        raise Exception("Function or modifier definition not found.")

    end_line = None
    brace_count = 0
    in_block = False

    for i in range(start_line, len(lines)):
        line = lines[i]
        brace_count += line.count('{')
        brace_count -= line.count('}')

        if brace_count > 0:
            in_block = True
        elif brace_count == 0 and in_block:
            end_line = i
            break

    if end_line is None:
        raise Exception("Function or modifier end not found.")

    code_block = "".join(lines[start_line:end_line + 1])
    return code_block


def find_function_slither(filename, start):
    with open(f"/content/smartbugs-wild/contracts/{filename}.sol", 'r') as file:
        file.seek(0)
        source_code = file.read()

    # Find the start of the function or modifier containing the given start position
    func_start_pos = max(source_code.rfind('function', 0, start), source_code.rfind('modifier', 0, start), source_code.rfind('constructor', 0, start))
    if func_start_pos == -1:
        return None

    # Find the end of the function or modifier using brace matching
    brace_count = 0
    in_block = False
    block_code = ''

    for pos in range(func_start_pos, len(source_code)):
        char = source_code[pos]
        block_code += char

        if char == '{':
            brace_count += 1
            in_block = True
        elif char == '}':
            brace_count -= 1
            if brace_count == 0 and in_block:
                break

    return block_code.strip()

def find_function_mythril(lines, lineno):

    # Adjust line number to 0-based index
    line_index = lineno - 1

    # Find the start of the function or modifier
    func_start = None
    for i in range(line_index, -1, -1):
        if 'function ' in lines[i] or 'constructor' in lines[i] or 'modifier ' in lines[i]:
            func_start = i
            break

    if func_start is None:
        return None

    # Find the end of the function or modifier using brace matching
    brace_count = 0
    in_block = False
    block_code = ''

    for i in range(func_start, len(lines)):
        line = lines[i]
        block_code += line

        brace_count += line.count('{')
        brace_count -= line.count('}')

        if brace_count > 0:
            in_block = True
        elif brace_count == 0 and in_block:
            break

    return block_code.strip()

# Example usage:
# lines = ["..."]  # Your Solidity source code lines
# lineno = 10  # Line number where the error was detected
# code_block = find_function_mythril(lines, lineno)
# print(code_block)


# filename = tool_sc[0][0]
# length =  19
# start = 83463

# function_code = find_function_slither(filename, start)
# print(f"Function extracted from {filename}:\n{function_code}\n")

# function_code = find_function_mythril(tool_sc[1][3], 28)
# print(f"Function extracted from {filename}:\n\n{function_code}\n")

# Crawling functions and source codes

In [ ]:
# Crawl all source codes and functions from those source codes to get dataset for each vulnerability (Each vulnerability will have some sub-vulnerabilites)

errors_dict = {
    're_entrancy': [
        ['reentrancy-benign', 'reentrancy-eth', 'reentrancy-unlimited-gas', 'reentrancy-no-eth'],
        ['external call to user-supplied','state change after external call', 'address external call to fixed address'],
        ['solidity_reentrancy']],
    'timestamp_dependency': [
        ['timestamp'], # slither
        ['dependence on predictable environment variable'], # mythril
        ['solidity_exact_time', 'vyper_timestamp_dependence']], # smartcheck
    'unhandled_exceptions': [
        ['unchecked-lowlevel', 'unchecked-send'],
        ['unchecked call return value'],
        ['solidity_unchecked_call']],
    'tx_origin': [
        ['tx-origin'],
        ['use of tx.origin'],
        ['solidity_tx_origin']]}

benign_functions = []

srcs = {
    'benign': [],
    're_entrancy': [],
    'timestamp_dependency': [],
    'unhandled_exceptions': [],
    'tx_origin': []
    }

funcs = {
    'benign': [],
    're_entrancy': [],
    'timestamp_dependency': [],
    'unhandled_exceptions': [],
    'tx_origin': []
    }

for k in errors_dict.keys():
    for j in (range(len(tool_list))):
        for i in tqdm(range(len(tool_sc[j]))):
            try:
                da_path = "/content/smartbugs-results/results/{}/icse20/{}/result.json"

                with open(da_path.format(tool_list[j], tool_sc[j][i])) as f:
                    data = json.load(f)

                with open(f"/content/smartbugs-wild/contracts/{tool_sc[j][i]}.sol", "r", encoding='utf-8') as f:
                    lines = f.readlines()
                    sol_file = "".join(lines)
                    sol_file = remove_comments(sol_file)
                    rc_lines = sol_file.splitlines(keepends=True)

                for q in range(len(rc_lines)):
                    if 'function' in rc_lines[q] or 'modifier' in rc_lines[q] or 'constructor' in rc_lines[q]:
                        function = find_function(rc_lines, q + 1)
                        if function.count('function') == 1 and function.count('Interface') == 0:
                            benign_functions.append(function.strip())

                if j == 0:
                    check = 0
                    if data['analysis'] != None and len(data['analysis']) != 0:
                        for issue in data['analysis']:
                            tmp = issue['check'].lower().strip()
                            if tmp in errors_dict[k][0]:
                                if check == 0:
                                    srcs[k].append((tool_sc[j][i], sol_file))
                                    check += 1

                                function = find_function_slither(tool_sc[j][i], issue['elements'][0]['source_mapping']['start']).strip()
                                function = remove_comments(function)
                                funcs[k].append(function)

                elif j == 1:
                    check = 0
                    if data['analysis'] != None and len(data['analysis']['issues']) != 0:
                        for issue in data['analysis']['issues']:
                            tmp = issue['title'].lower().strip()
                            if tmp in errors_dict[k][1]:
                                if check == 0:
                                    srcs[k].append((tool_sc[j][i], sol_file))
                                    check += 1

                                function = find_function_mythril(lines, issue['lineno']).strip()
                                function = remove_comments(function)
                                funcs[k].append(function)
                else:
                    check = 0
                    if data['analysis'] != None and len(data['analysis']) != 0:
                        for issue in data['analysis']:
                            tmp = issue['name'].lower().strip()
                            if tmp in errors_dict[k][2]:
                                if check == 0:
                                    srcs[k].append((tool_sc[j][i], sol_file))
                                    check += 1

                                function = find_function_smartcheck(lines, issue['line']).strip()
                                function = remove_comments(function)
                                funcs[k].append(function)
            except Exception as e:
                pass

100%|██████████| 47557/47557 [01:38<00:00, 481.27it/s]


In [ ]:
# Get all source codes that are identified as benign

benign_srcs = tool_sc[0] + tool_sc[1] + tool_sc[2]
benign_srcs = list(set(benign_srcs))

re_srcs = [u for u, v in srcs['re_entrancy']]
td_srcs = [u for u, v in srcs['timestamp_dependency']]
ue_srcs = [u for u, v in srcs['unhandled_exceptions']]
tx_srcs = [u for u, v in srcs['tx_origin']]

benign_srcs = list(set(benign_srcs) - set(re_srcs) - set(td_srcs) - set(ue_srcs) - set(tx_srcs))
print(len(benign_srcs))
for src_name in benign_srcs:
    try:
        with open(f"/content/smartbugs-wild/contracts/{src_name}.sol", "r", encoding='utf-8') as f:
            lines = f.readlines()
            sol_file = "".join(lines)
            sol_file = remove_comments(sol_file)

        srcs['benign'].append((src_name, sol_file))
    except Exception as e:
        pass

print(len(srcs['benign']))

36397
36234


In [ ]:
len(set(benign_functions))

240339

In [ ]:
# Filter out all benign functions and remove duplicates

funcs['benign'] = list(set(benign_functions) - set(funcs['re_entrancy']) - set(funcs['timestamp_dependency']) - set(funcs['unhandled_exceptions']) - set(funcs['tx_origin']))
funcs['benign'] = list(set(funcs['benign']))
funcs['re_entrancy'] = list(set(funcs['re_entrancy']))
funcs['timestamp_dependency'] = list(set(funcs['timestamp_dependency']))
funcs['unhandled_exceptions'] = list(set(funcs['unhandled_exceptions']))
funcs['tx_origin'] = list(set(funcs['tx_origin']))

print(len(srcs['benign']), len(srcs['re_entrancy']), len(srcs['timestamp_dependency']), len(srcs['unhandled_exceptions']), len(srcs['tx_origin']))
print(len(funcs['benign']), len(funcs['re_entrancy']), len(funcs['timestamp_dependency']), len(funcs['unhandled_exceptions']), len(funcs['tx_origin']))

In [ ]:
# 36184 8405 2461 2163 671
# 226420 9699 2592 1443 430

In [ ]:
funcs_mapping = {
    'benign': 0,
    're_entrancy': 1,
    'timestamp_dependency': 2,
    'unhandled_exceptions': 3,
    'tx_origin': 4
    }

funcs_list = []
funcs_labels = []
funcs_enlabels = []

for key in funcs.keys():
    funcs_list.extend(funcs[key])
    funcs_labels.extend([key] * len(funcs[key]))
    funcs_enlabels.extend([funcs_mapping[key]] * len(funcs[key]))

funcs_df = pd.DataFrame({'func': funcs_list, 'label': funcs_labels, 'enlabel': funcs_enlabels})
pd.set_option('display.max_colwidth', None)

funcs_df[funcs_df['func'] == '']
# funcs_df.drop(funcs_df[funcs_df['func'] == ''].index, inplace=True)

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
if not os.path.exists('/content/drive/MyDrive/ScamSolidityCodeDetection/SolFunctionDataset'):
    os.mkdir('/content/drive/MyDrive/ScamSolidityCodeDetection/SolFunctionDataset')

if not os.path.exists('/content/drive/MyDrive/ScamSolidityCodeDetection/SolCodeDataset'):
    os.mkdir('/content/drive/MyDrive/ScamSolidityCodeDetection/SolCodeDataset')

In [ ]:
funcs_df.to_csv('/content/drive/MyDrive/ScamSolidityCodeDetection/SolFunctionDataset/funcs_ver2.csv', index=False)

In [ ]:
print(len(os.listdir('/content/drive/MyDrive/ScamSolidityCodeDetection/SolCodeDataset/Benign')))

13994


In [ ]:
# We have too many benign files so we only sample a resonable number of them

seed = 42
random.seed(seed)

def write_file(sol_info, dest):
  """
    This function create in order to help changing collected
    data from csv such as: Src Address, Src Code to File
    - sol_info (tuple): (sol_name, sol_code)
    - dest (str): Destination to the .sol saved file
  """
  try:
    sol_name, sol_code = sol_info
    with open(f'{dest}/{sol_name}.sol', 'w') as f:
      f.write(sol_code)
    print('Finish writting SOL file:', f'{dest}/{sol_name}.sol')
  except Exception as e:
    print("Error writting file:", str(e))


for ftype in ['Benign']:
    # Example usage
    sol_list = random.sample(srcs[ftype.lower()], 10000)
    # sol_list = srcs[ftype.lower()]
    solFileDst = f'/content/drive/MyDrive/ScamSolidityCodeDetection/SolCodeDataset/{ftype}'
    if os.path.exists(solFileDst) == False:
        os.mkdir(solFileDst)

    solFileInstall = [u for u, v in sol_list]
    solFileInstalled = [file.rstrip(".sol") for file in os.listdir(solFileDst) if file.endswith(".sol")]
    solFiles = list((set(solFileInstall) - set(solFileInstalled)))

    print(len(solFileInstall), len(solFileInstalled), len(solFiles))
    # pool = ThreadPool(8)
    # pool.map(partial(write_file, dest=solFileDst), sol_list)

10000 13994 6251


# EDA

In [ ]:
# @title Check vulnerability types in each tool found in this dataset
mythril_list, slither_list, smartcheck_list = [], [], []
names = ['slither', 'mythril', 'smartcheck']

with open("/content/smartbugs-results/metadata/results_wild.json", "r") as f:
    re_data = json.load(f)

for key, value in tqdm(re_data.items()):
    mythril_list.extend([k.lower() for k in re_data[key]['tools'][names[1]]['vulnerabilities'].keys()])
    slither_list.extend([k.lower() for k in re_data[key]['tools'][names[0]]['vulnerabilities'].keys()])
    smartcheck_list.extend([k.lower() for k in re_data[key]['tools'][names[2]]['vulnerabilities'].keys()])

mythril_list = list(set(mythril_list))
slither_list = list(set(slither_list))
smartcheck_list = list(set(smartcheck_list))

print(slither_list)
print('-' * 1000)
print(mythril_list)
print('-' * 1000)
print(smartcheck_list)

['arbitrary-send', 'timestamp', 'uninitialized-storage', 'reentrancy-eth', 'reentrancy-no-eth', 'uninitialized-state', 'calls-loop', 'tx-origin', 'suicidal', 'locked-ether', 'controlled-delegatecall', 'incorrect-equality', 'uninitialized-local', 'low-level-calls', 'reentrancy-benign', 'unused-return']
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [ ]:
# @title Check number of functions contain a pre-defined keyword in our dataset
from datasets import load_dataset

# Load the dataset from Hugging Face I used to upload
dataset = load_dataset("Quangnguyen711/solidity_re_entrancy_dataset")

# Convert to a dataframe for easy manipulation
df = dataset['train'].to_pandas()

# Filter rows where label == 1 - Which is reentrancy
reentrancy_vulnerable_funcs = df[df['label'] == 1]

# Count how many of those functions contain the keyword 'payable'
c_count = reentrancy_vulnerable_funcs['func'].str.contains('public payable').sum()
p_count = reentrancy_vulnerable_funcs['func'].str.contains('external payable').sum()
payable_count = c_count + p_count

# Total number of functions with label 1
total_vulnerable_funcs = reentrancy_vulnerable_funcs.shape[0]

# Output the results
print(f"Total functions labeled with reentrancy vulnerability: {total_vulnerable_funcs}")
print(f"Functions containing 'payable': {payable_count}")

Total functions labeled with reentrancy vulnerability: 8726
Functions containing 'payable': 760


In [ ]:
# @title Check number of functions contain a pre-defined keyword in dataset use to crawl

errors_dict = {
    're_entrancy': [
        ['reentrancy-benign', 'reentrancy-eth', 'reentrancy-unlimited-gas', 'reentrancy-no-eth'],
        ['external call to user-supplied','state change afer external call', 'address external call to fixed address'],
        ['solidity_reentrancy']]}

benign_functions = []
cnt = 0
srcs = {
    're_entrancy': []
    }

funcs = {
    're_entrancy': []
    }

for k in errors_dict.keys():
    for j in (range(len(tool_list))):
        for i in tqdm(range(len(tool_sc[j]))):
            try:
                da_path = "/content/smartbugs-results/results/{}/icse20/{}/result.json"

                with open(da_path.format(tool_list[j], tool_sc[j][i])) as f:
                    data = json.load(f)

                with open(f"/content/smartbugs-wild/contracts/{tool_sc[j][i]}.sol", "r", encoding='utf-8') as f:
                    lines = f.readlines()
                    sol_file = "".join(lines)
                    sol_file = remove_comments(sol_file)
                    rc_lines = sol_file.splitlines(keepends=True)

                for q in range(len(rc_lines)):
                    if 'function' in rc_lines[q] or 'modifier' in rc_lines[q] or 'constructor' in rc_lines[q]:
                        function = find_function(rc_lines, q + 1)
                        if function.count('function') == 1 and function.count('Interface') == 0:
                            benign_functions.append(function.strip())

                if j == 0:
                    check = 0
                    if data['analysis'] != None and len(data['analysis']) != 0:
                        for issue in data['analysis']:
                            tmp = issue['check'].lower().strip()
                            if tmp in errors_dict[k][0]:
                                if check == 0:
                                    srcs[k].append((tool_sc[j][i], sol_file))
                                    check += 1

                                function = find_function_slither(tool_sc[j][i], issue['elements'][0]['source_mapping']['start']).strip()
                                function = remove_comments(function)
                                # print(function)
                                # print("-" * 100)
                                funcs[k].append(function)
                                # print(len(funcs[k]))

                elif j == 1:
                    check = 0
                    if data['analysis'] != None and len(data['analysis']['issues']) != 0:
                        for issue in data['analysis']['issues']:
                            tmp = issue['title'].lower().strip()
                            if tmp in errors_dict[k][1]:
                                print(issue['name'])
                                if check == 0:
                                    srcs[k].append((tool_sc[j][i], sol_file))
                                    check += 1

                                function = find_function_mythril(lines, issue['lineno']).strip()

                                function = remove_comments(function)
                                funcs[k].append(function)
                                print(len(funcs[k]))
                else:
                    check = 0
                    if data['analysis'] != None and len(data['analysis']) != 0:
                        for issue in data['analysis']:
                            tmp = issue['name'].lower().strip()
                            if tmp in errors_dict[k][2]:
                                print(issue['name'])
                                if check == 0:
                                    srcs[k].append((tool_sc[j][i], sol_file))
                                    check += 1

                                function = find_function_smartcheck(lines, issue['line']).strip()

                                function = remove_comments(function)
                                funcs[k].append(function)
                                print(len(funcs[k]))
            except Exception as e:
                cnt += 1
                pass

# Count how many of those functions contain the keyword 'payable'
c_count = sum([1 if "public" in st and "payable" in st else 0 for st in funcs['re_entrancy']])
p_count = sum([1 if "external" in st and "payable" in st else 0 for st in funcs['re_entrancy']])
payable_count = c_count + p_count

# Total number of functions with label 1
total_vulnerable_funcs = len(funcs['re_entrancy'])

# Output the results
print(f"Total functions labeled with reentrancy vulnerability: {total_vulnerable_funcs}")
print(f"Functions containing tag: {payable_count}")

Total functions labeled with reentrancy vulnerability: 31885
Functions containing tag: 4196
